# Phase 3 — Feature Engineering Check
Create lag and rolling window features using PySpark Window functions.
Uses 1% sample to fit in local memory.

In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"D:\Retail Demand Forecasting\.venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"D:\Retail Demand Forecasting\.venv\Scripts\python.exe"
os.environ["SPARK_LOCAL_DIRS"] = r"D:\spark-temp"
os.environ["HADOOP_HOME"] = r"D:\hadoop"

os.chdir(r"D:\Retail Demand Forecasting")

In [2]:
from pyspark.sql import SparkSession
from retail_demand_forecasting.nodes.data_engineering import unpivot_sales
from retail_demand_forecasting.nodes.feature_engineering import create_features

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("phase3_feature_engineering")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .config("spark.local.dir", r"D:\spark-temp")
    .getOrCreate()
)
print(f"Spark {spark.version} ready.")

Spark 4.1.1 ready.


## 1. Load raw datasets, unpivot, and sample 1%

In [3]:
PROJECT_ROOT = r"D:\Retail Demand Forecasting"

sales_train_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(os.path.join(PROJECT_ROOT, "data/01_raw/sales_train_validation.csv"))
calendar_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(os.path.join(PROJECT_ROOT, "data/01_raw/calendar.csv"))
sell_prices_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(os.path.join(PROJECT_ROOT, "data/01_raw/sell_prices.csv"))

melted_df = unpivot_sales(sales_train_raw, calendar_raw, sell_prices_raw)
# Sample 1% to fit in local memory
melted_df = melted_df.sample(fraction=0.01, seed=42)
print(f"Melted (1% sample): ~{melted_df.count():,} rows")

Melted (1% sample): ~583,733 rows


## 2. Run create_features node

In [4]:
params = {
    "lag_days": [7, 28],
    "rolling_window_days": [7, 28],
}

featured_df = create_features(melted_df, params)
print(f"Featured: ~{featured_df.count():,} rows x {len(featured_df.columns)} columns")

Featured: ~1,417 rows x 27 columns


## 3. Schema

In [5]:
featured_df.printSchema()

root
 |-- day_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- dept_id: string (nullable = true)
 |-- cat_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- state_id: string (nullable = true)
 |-- sales: integer (nullable = true)
 |-- sell_price: double (nullable = true)
 |-- wm_yr_wk: integer (nullable = true)
 |-- event_name_1: string (nullable = true)
 |-- event_type_1: string (nullable = true)
 |-- event_name_2: string (nullable = true)
 |-- event_type_2: string (nullable = true)
 |-- snap_CA: integer (nullable = true)
 |-- snap_TX: integer (nullable = true)
 |-- snap_WI: integer (nullable = true)
 |-- lag_7: integer (nullable = true)
 |-- lag_28: integer (nullable = true)
 |-- rolling_mean_7: double (nullable = true)
 |-- rolling_mean_28: double (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nul

## 4. Sample rows with features

In [6]:
featured_df.select(
    "day_id", "date", "item_id", "store_id", "sales",
    "lag_7", "lag_28", "rolling_mean_7", "rolling_mean_28",
    "day_of_week", "month", "has_event_1",
).limit(10).toPandas()

,day_id,date,item_id,store_id,sales,lag_7,lag_28,rolling_mean_7,rolling_mean_28,day_of_week,month,has_event_1
0,1889,2016-03-31,FOODS_1_034,CA_1,0,0,0,0.142857,0.250000,5,3,0
1,1784,2015-12-17,FOODS_1_087,CA_1,4,0,0,2.571429,1.678571,5,12,0
2,1820,2016-01-22,FOODS_1_087,CA_1,2,7,0,3.142857,1.821429,6,1,0
3,1870,2016-03-12,FOODS_1_087,CA_1,1,2,0,2.428571,1.892857,7,3,0
4,1907,2016-04-18,FOODS_2_089,CA_1,1,3,0,0.714286,0.178571,2,4,0
5,1811,2016-01-13,FOODS_2_254,CA_1,0,0,0,0.000000,0.035714,4,1,0
6,1516,2015-03-24,FOODS_2_313,CA_1,0,0,0,0.000000,0.321429,3,3,0
7,1519,2015-03-27,FOODS_2_313,CA_1,0,0,0,0.000000,0.321429,6,3,0
8,1547,2015-04-24,FOODS_2_313,CA_1,0,0,1,0.000000,0.321429,6,4,0
9,1668,2015-08-23,FOODS_2_313,CA_1,0,0,0,0.000000,0.285714,1,8,0


## 5. Feature statistics

In [7]:
featured_df.select(
    "sales", "lag_7", "lag_28", "rolling_mean_7", "rolling_mean_28",
).describe().toPandas()

,summary,sales,lag_7,lag_28,rolling_mean_7,rolling_mean_28
0,count,1417,1417,1417,1417,1417
1,mean,1.7367678193366267,1.347212420606916,1.036697247706422,1.4927916120576654,1.3992337937292079
2,stddev,6.366558019413388,5.195824288276871,4.605334335133668,4.90909617071535,5.266487953879067
3,min,0,0,0,0.0,0.0
4,max,98,95,64,63.142857142857146,69.60714285714286


In [8]:
spark.stop()
print("Phase 3 complete.")

Phase 3 complete.
